# Ticket 8: document-level reconciliation with bucket + cause

Scope: company 1000, FY2024 P01 (the only period `fact_gl_line` covers
so far). Issue #8: FULL OUTER JOIN `stg_gl` to `fact_gl_line` on the
grain, classify every difference with one bucket + one cause.

In [1]:
import duckdb

con = duckdb.connect("../warehouse.duckdb", read_only=True)
SCOPE = "company_code = 1000 AND fiscal_year = 2024 AND fiscal_period = 1"

In [2]:
# literal reading: FULL OUTER JOIN stg_gl to the full fact_gl_line table
con.execute(f"""
    WITH s AS (SELECT * FROM stg_gl WHERE {SCOPE}),
         f AS (SELECT * FROM fact_gl_line)
    SELECT
        CASE WHEN s.document_id IS NULL THEN 'missing_in_stg'
             WHEN f.document_id IS NULL THEN 'missing_in_fact'
             WHEN ABS(COALESCE(s.local_amount,0) - COALESCE(f.local_amount,0)) > 0.01 THEN 'amount_changed'
             ELSE 'matched' END AS bucket,
        COUNT(*) n
    FROM s FULL OUTER JOIN f USING (company_code, document_id, line_number, fiscal_year, fiscal_period)
    GROUP BY 1
""").fetchall()

[('matched', 13140), ('missing_in_fact', 2)]

`missing_in_fact=2, matched=13140`. A literal row-for-row join against
the full `fact_gl_line` table only ever produces `missing_in_fact`
(ticket 5's 1 excluded unbalanced document). `opening_balance` and
`is_post_close` rows are present, identical, in both tables - ticket 4
loads and flags them, never excludes them from the table - so they show
as perfectly matched here, not as a mismatch. `intentionally_excluded`
never fires under this reading. 3 of the 4 required buckets
(`missing_in_stg`, `amount_changed`, `intentionally_excluded`) would be
permanently empty, not just empty this period.

In [3]:
# second reading: compare stg_gl to the CLOSE-ELIGIBLE subset of fact_gl_line
# (excludes rows flagged is_opening_balance / is_closing_entry / is_post_close,
# the ones business-rules.md says are "never mixed into in-period totals")
con.execute(f"""
    WITH s AS (SELECT * FROM stg_gl WHERE {SCOPE}),
         f AS (SELECT * FROM fact_gl_line WHERE NOT is_opening_balance AND NOT is_closing_entry AND NOT is_post_close)
    SELECT
        CASE WHEN s.document_id IS NULL THEN 'missing_in_stg'
             WHEN f.document_id IS NULL THEN 'excluded_or_missing'
             WHEN ABS(COALESCE(s.local_amount,0) - COALESCE(f.local_amount,0)) > 0.01 THEN 'amount_changed'
             ELSE 'matched' END AS bucket,
        COUNT(*) n
    FROM s FULL OUTER JOIN f USING (company_code, document_id, line_number, fiscal_year, fiscal_period)
    GROUP BY 1
""").fetchall()

[('matched', 12923), ('excluded_or_missing', 219)]

`matched=12923, excluded_or_missing=219`. Splitting `excluded_or_missing`
by why the row isn't close-eligible:

In [4]:
con.execute(f"""
    WITH s AS (SELECT * FROM stg_gl WHERE {SCOPE}),
         f AS (SELECT * FROM fact_gl_line WHERE NOT is_opening_balance AND NOT is_closing_entry AND NOT is_post_close)
    SELECT raw.is_opening_balance, raw.is_closing_entry, raw.is_post_close,
           (raw.document_id IS NOT NULL) AS present_in_full_fact, COUNT(*) n
    FROM s
    LEFT JOIN f USING (company_code, document_id, line_number, fiscal_year, fiscal_period)
    LEFT JOIN fact_gl_line raw USING (company_code, document_id, line_number, fiscal_year, fiscal_period)
    WHERE f.document_id IS NULL
    GROUP BY 1, 2, 3, 4
""").fetchall()

[(None, None, None, False, 2),
 (False, False, True, True, 200),
 (True, False, False, True, 17)]

Three groups, and every one cross-validates against an earlier ticket's
own finding:

- **17 rows**, `is_opening_balance=true`, present in the full
  `fact_gl_line` table - matches mission 04's 17 `OPENING_BALANCE` rows
  exactly. Real row, excluded from the close-eligible view by design.
  `bucket=intentionally_excluded, cause=opening_balance`.
- **200 rows**, `is_post_close=true`, present in the full table - matches
  mission 04's exploration (`is_post_close=200`) exactly.
  `bucket=intentionally_excluded, cause=post_close`.
- **2 rows**, no flags, *not* present in the full `fact_gl_line` table at
  all - the 1 unbalanced document (2 lines) mission 04/05 already
  exclude. `bucket=missing_in_fact, cause=unbalanced_document`.

17 + 200 + 2 = 219, the exact count from the cell above. Every row is
explained; nothing falls to `unknown`.

In [5]:
# is the 1 unbalanced document also part of a reversal pair? (checking whether
# reversal_pair cause has a real trigger in this period's data at all)
con.execute("""
    SELECT * FROM recon_reversal_pairs
    WHERE original_document_id = '6a00ef40-e8ec-4c9d-89b1-2c3440301a67'
       OR reversal_document_id = '6a00ef40-e8ec-4c9d-89b1-2c3440301a67'
""").fetchall()

[]

**Empty.** No overlap. The one document this ticket's join actually
excludes has nothing to do with any detected reversal pair. `CL`
(closing entry) also has zero real rows this period (no `CL`
`document_type` found in earlier exploration). Two of the five
`intentionally_excluded` causes (`closing_entry`, `reversal_pair`) have
no real case in P01 to derive their trigger condition from - unlike
`opening_balance`/`post_close`, which map straight onto an existing
boolean flag, there's no `is_reversal_pair`-style column to key
`reversal_pair` off of. This needs a design decision, not just a
synthetic-row proof of arithmetic (see the mission's before-build items).